# Small-Scale Analysis

In this tutorial, we will cover strategies for working with a small slice of a large catalog before committing to a full-scale computation:
- narrow the sky area with a spatial filter (e.g., `cone_search`)
- inspect a single partition with `.partitions[i]`
- filter rows to a manageable subset (e.g., bright stars)
- peek at the first few rows of every partition with `map_partitions`
- draw a random sample with `.random_sample()`

## Introduction

Large astronomical catalogs can contain billions of rows spread across thousands of partitions. Running a pipeline on the full dataset is expensive, and it is easy to waste hours on a bug that could have been caught in seconds on a small slice.

A good workflow starts small:

1. **Narrow the sky**: work only in the patch of sky you actually care about.
2. **Inspect one partition**: confirm the data looks right before processing everything.
3. **Filter aggressively**: drop rows you do not need as early as possible.
4. **Peek at multiple partitions**: cheaply verify your function behaves correctly across partition boundaries.
5. **Draw a random sample**: get a statistically representative preview without a full compute.

Each technique in this tutorial reduces the amount of data you touch, so you can iterate quickly and scale up only once you are confident in the result.

In [ ]:
import pandas as pd

import lsdb
from dask.distributed import Client

## 1. Open a catalog

In [ ]:
client = Client(n_workers=4, memory_limit="auto")

We open the ZTF DR14 object catalog. The catalog is loaded lazily — no row data is read yet.

In [ ]:
ztf_object_path = "https://data.lsdb.io/hats/ztf_dr14/ztf_object"
ztf_object = lsdb.open_catalog(ztf_object_path)
ztf_object

## 2. Region selection

The simplest way to reduce the amount of data you work with is to restrict the sky area. A `cone_search` keeps only the partitions that overlap a circle defined by a center `(ra, dec)` and `radius_arcsec`.

Starting with a small cone lets you develop and test your pipeline on a tiny fraction of the catalog. Once the pipeline is correct, you can widen the cone or remove it entirely.

In [ ]:
ztf_cone = ztf_object.cone_search(ra=200.0, dec=30.0, radius_arcsec=1 * 36_000)
ztf_cone

The `npartitions` has dropped from thousands to a handful, so every subsequent step is much cheaper.

We can also use a pre-built `ConeSearch` object, which lets us reuse the same region across multiple catalogs.

In [ ]:
from lsdb import ConeSearch

cone = ConeSearch(ra=200.0, dec=30.0, radius_arcsec=1 * 36_000)
ztf_cone = ztf_object.search(cone)
ztf_cone

## 3. Partition selection

Even within a small region you may want to look at a single partition in isolation. Use `.partitions[i]` to index into the catalog by partition number. The result is a lazy Dask DataFrame for that one partition.

This is useful when you want to call `.compute()` on just one chunk to inspect values or test a function without touching the rest of the catalog.

In [ ]:
# Look at the first partition
first_partition = ztf_cone.partitions[0]
first_partition.compute()

You can find the HEALPix pixel that corresponds to a given partition index using `get_healpix_pixels()`.

In [ ]:
pixels = ztf_cone.get_healpix_pixels()
print(f"Partition 0 covers: {pixels[0]}")

## 4. Sub-filtering

Row filters let you trim the data further before any expensive computation. For example, selecting only bright stars reduces the number of rows dramatically and gives you a representative but manageable subset to work with.

In [ ]:
ztf_cone_and_bright = ztf_cone.query("mean_mag_g < 16")
ztf_cone_and_bright

In [ ]:
ztf_cone_and_bright.head(5)

Filters compose: you can chain a spatial filter with a row filter and LSDB will push both into the same pipeline.

## 5. Peeking at every partition

`map_partitions` applies a function to each partition individually. Passing `pd.DataFrame.head` (or a small wrapper around it) is a cheap way to fetch the first few rows of every partition without loading the full catalog into memory.

This is especially useful for checking that a transformation produces the expected columns and values across all partition boundaries.

In [ ]:
# Grab the first 3 rows from each partition, then compute
sample_per_partition = ztf_cone.map_partitions(lambda df: df.head(3))
sample_per_partition.compute()

You can pass `pd.DataFrame.head` directly as the function, along with `n` as an extra keyword argument.

In [ ]:
sample_per_partition = ztf_cone.map_partitions(pd.DataFrame.head, n=3)
sample_per_partition.compute()

## 6. Random sample

`.random_sample(n)` draws approximately `n` rows distributed proportionally across all partitions. Unlike `.head()`, which always returns rows from the first partitions, a random sample is representative of the whole catalog.

Use `.random_sample()` when you need a statistical cross-section of the data — for example, to estimate a distribution or spot-check the output of a filter.

Pass a `seed` for reproducible results.

In [ ]:
sample = ztf_cone.random_sample(n=20, seed=42)
sample

If you only want to sample from a single partition, use `.sample(partition_id, n)` instead. This avoids touching any other partition.

In [ ]:
single_partition_sample = ztf_cone.sample(partition_id=0, n=5, seed=42)
single_partition_sample

## Closing the Dask client

In [ ]:
client.close()

## About

**Authors**: Olivia Lynn

**Last updated on**: May 18, 2026

If you use `lsdb` for published research, please cite following [instructions](https://docs.lsdb.io/en/stable/citation.html).